## Decoder Layer

In [1]:
import torch
import torch.nn as nn

class DecoderLayer(nn.Module):

    def __init__(self,d_model,heads,d_ff):

        super().__init__()

        self.masked_attn = nn.MultiheadAttention(
            d_model,heads,batch_first=True)

        self.norm1 = nn.LayerNorm(d_model)

        self.cross_attn = nn.MultiheadAttention(
            d_model,heads,batch_first=True)

        self.norm2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.ReLU(),
            nn.Linear(d_ff,d_model)
        )

        self.norm3 = nn.LayerNorm(d_model)

    def forward(self,x,enc):

        attn,_ = self.masked_attn(x,x,x)

        x = self.norm1(x+attn)

        cross,_ = self.cross_attn(x,enc,enc)

        x = self.norm2(x+cross)

        ffn = self.ffn(x)

        x = self.norm3(x+ffn)

        return x


# TEST
x = torch.rand(2,5,16)
enc = torch.rand(2,5,16)

decoder = DecoderLayer(16,4,64)

out = decoder(x,enc)

print("Output:",out.shape)
print(out[0])

Output: torch.Size([2, 5, 16])
tensor([[ 0.4325, -1.3828, -0.5691, -0.1673, -1.3945,  1.1650,  1.4823, -0.8944,
          0.0240, -1.1862, -0.7996, -0.2861,  0.8554, -0.1428,  1.7981,  1.0657],
        [-0.0286,  0.6997, -0.9502,  0.4289, -1.1356,  0.7254,  1.8732, -0.1276,
          1.1074, -1.5095,  0.0708, -0.1191, -0.8902,  1.6665, -0.4885, -1.3226],
        [-0.0663,  0.5259, -0.5546,  0.5511,  1.2789,  0.1161,  0.1370, -2.0214,
         -1.5707,  0.7194, -1.7183,  0.4396,  1.4497, -0.4832,  0.8913,  0.3055],
        [-0.6977, -0.4729,  1.1296, -2.1207,  0.2106, -0.3170,  0.9401, -0.8058,
          0.2450, -1.7195,  0.0560, -0.0415,  0.9532,  1.7315,  0.9532, -0.0442],
        [-1.1192, -0.4339, -0.5720, -0.7159,  0.4261, -1.0166, -0.5075,  0.2590,
         -0.2410, -1.9496,  1.4187,  0.1839,  2.0512,  0.9513,  0.2079,  1.0576]],
       grad_fn=<SelectBackward0>)
